# Final ConvNeXt pipeline: 224 + 320, no TTA

**Selected run: Kaggle public score 0.95039.** The saved submission and run
metadata are in `results/`; a new training run may differ.

Start from the attached 224 backup. To recreate those checkpoints from scratch,
see `archive/experiments/cnn_2.ipynb` and `archive/experiments/cnn_2_cv.ipynb`.
The unchanged original EDA and Validation notebooks remain at repository root.

Target an approximately two-hour session without changing `convnext_tiny.fb_in1k`,
labels, original grouped folds, augmentation or metric definitions. This is a NEW
experiment, not an exact reproduction of the previous 0.94996 submission.

Attach the original `convnext_backup` Dataset containing `convnext_cv` AND
`convnext_cv_ensemble`. All five matching 224 checkpoints and their saved test
probabilities are required; no 224 training or test inference is repeated.

Changes: at most FOUR epochs per fold, EMA-only epoch validation, channels-last
on CUDA, AMP, no flip inference anywhere, and reuse of the exact validation
probabilities stored with the best checkpoint. Epoch 0 remains a candidate.
The effective batch remains 32 (16 x 2 accumulation) and NUM_WORKERS stays 0.

Training has a SOFT 95-minute allowance, shared dynamically across the remaining
folds. Stop decisions happen only at epoch boundaries and every unfinished fold
gets at least one full epoch. Keep 25 minutes for test inference and saving.
Slow GPU/I/O, startup, W&B and downloading the archive can exceed this allowance;
this is not a hard two-hour guarantee. Fewer epochs and skipping raw validation
can change the best checkpoint and score. cuDNN autotuning can change numerical
reproducibility across runs. Backbone, split and validation calculations are unchanged.

Run this one notebook top to bottom: five-fold training -> OOF calibration ->
original-only test inference -> equal 224/320 blend -> compact inference backup.
`last.pt` keeps full current-session resume state. The compact ZIP deliberately
contains best weights and inference metadata, not optimizer states; fully completed
folds can be restored, while an unfinished fold needs its `last.pt` to resume.

After each completed fold the compact archive is refreshed. Files in `/kaggle/working`
are still temporary: DOWNLOAD the ZIP and/or save a Kaggle version including Output
before ending the session. The archive alone does not upload itself anywhere.

In [ ]:
# Optional, separate install cell; restart the kernel afterwards.
# %pip install -q timm wandb "albumentations>=2,<3"

In [ ]:
import json
import os
import random
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

print('torch:', torch.__version__, '| timm:', timm.__version__)
import hashlib
import shutil
from time import perf_counter
import copy
import math

torch: 2.11.0+cu128 | timm: 1.0.29


## 1. Short-run configuration
Same-fold 224 initialization, four-epoch cap, EMA checkpoint selection, and a soft
training budget. Change settings only before starting; use a new output directory
for incompatible settings. This notebook does not modify the old runs.

In [ ]:
SEED = 21
MODEL_NAME = 'convnext_tiny.fb_in1k'
LABEL_COLS = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
DATA_DIR = Path('/kaggle/input/competitions/cats-and-dogs')
FOLDS_PATH = None
CV_DIR = Path('/kaggle/working/convnext_cv_320_fast')
SOURCE_CV_DIR = Path('/kaggle/working/convnext_cv')
PREVIOUS_CV_DIR = None  # saved convnext_cv_320 from an earlier session
FOLDS_TO_TRAIN = [0, 1, 2, 3, 4]
IMAGE_SIZE = 320
BATCH_SIZE = 16
ACCUMULATION_STEPS = 2
EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2
LR = 1e-5
HEAD_LR = 5e-5
WARMUP_EPOCHS = 1
EMA_DECAY = 0.99
MIN_LR = 2e-7
WEIGHT_DECAY = 1e-2
THRESHOLD = 0.5
NUM_WORKERS = 0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
USE_WANDB = True
WANDB_PROJECT = 'cats-and-dogs'
WANDB_ENTITY = 'yrachynska-kyiv-school-of-economics'  # shared team project
WANDB_MODE = 'online'
WANDB_LOG_EVERY = 40
TRAIN_BUDGET_MINUTES = 95
POSTPROCESS_RESERVE_MINUTES = 25  # planning reserve, not a hard execution timeout
CHANNELS_LAST = DEVICE.type == 'cuda'
MEMORY_FORMAT = torch.channels_last if CHANNELS_LAST else torch.contiguous_format
RUN_STARTED = perf_counter()
TRAIN_DEADLINE = RUN_STARTED + TRAIN_BUDGET_MINUTES * 60
SOURCE_PREDICTIONS_DIR = Path('/kaggle/working/convnext_cv_ensemble')
TEST_OUTPUT_DIR = Path('/kaggle/working/convnext_cv_320_fast_ensemble')
BLEND_OUTPUT_DIR = Path('/kaggle/working/convnext_blend_224_320_fast')
BACKUP_PATH = Path('/kaggle/working/convnext_320_fast_inference_backup.zip')
assert len(FOLDS_TO_TRAIN) == len(set(FOLDS_TO_TRAIN)) and set(FOLDS_TO_TRAIN).issubset(range(5))
assert EPOCHS > 0 and BATCH_SIZE > 0 and EARLY_STOPPING_PATIENCE > 0
assert ACCUMULATION_STEPS > 0 and 0 <= EMA_DECAY < 1 and 0 <= WARMUP_EPOCHS < EPOCHS
assert HEAD_LR >= LR > 0 and IMAGE_SIZE % 32 == 0
assert 0 <= MIN_LR <= LR and NUM_WORKERS == 0
assert CV_DIR.resolve() != SOURCE_CV_DIR.resolve(), 'Source and destination must differ.'
CV_DIR.mkdir(parents=True, exist_ok=True)
preprocessing = {'name': 'albumentations_resize_v1', 'image_size': IMAGE_SIZE,
                 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225], 'interpolation': 1}
print('device:', DEVICE, '| requested folds:', FOLDS_TO_TRAIN, '| outputs:', CV_DIR)
assert TRAIN_BUDGET_MINUTES > 0 and POSTPROCESS_RESERVE_MINUTES >= 0


device: cuda | requested folds: [0, 1, 2, 3, 4] | outputs: /kaggle/working/convnext_cv_320_fast


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = DEVICE.type == 'cuda'
    torch.backends.cudnn.deterministic = DEVICE.type != 'cuda'

## 2. Load the original split

Validate IDs, labels, five-fold assignments and group separation. Fingerprint the exact CSV contents to reject stale checkpoints or changed splits.

In [ ]:
def digest_text(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()

def file_digest(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()

def save_json(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2))
    temporary.replace(path)

def save_torch(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


if not (SOURCE_CV_DIR / 'ensemble_config.json').is_file():
    candidates = []
    for directory, subdirs, files in os.walk('/kaggle/input'):
        subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
        if {'ensemble_config.json', 'cv_config.json', 'folds.csv', 'oof_predictions.csv'}.issubset(files):
            cfg = json.loads((Path(directory) / 'cv_config.json').read_text())
            if cfg.get('pipeline') == 'convnext_cv_v1' and cfg['preprocessing']['image_size'] == 224:
                candidates.append(Path(directory))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Attach the complete original convnext_cv output or set SOURCE_CV_DIR. Candidates: {candidates}')
    SOURCE_CV_DIR = candidates[0]
assert CV_DIR.resolve() != SOURCE_CV_DIR.resolve()
source_manifest = json.loads((SOURCE_CV_DIR / 'ensemble_config.json').read_text())
assert source_manifest['model_name'] == MODEL_NAME and source_manifest['label_cols'] == LABEL_COLS
assert source_manifest['folds'] == list(range(5))
assert source_manifest['preprocessing']['image_size'] == 224
for key in ['name', 'mean', 'std', 'interpolation']:
    assert source_manifest['preprocessing'][key] == preprocessing[key]
for f in range(5):
    assert file_digest(SOURCE_CV_DIR / f'fold_{f}' / 'best.pt') == source_manifest['checkpoint_hashes'][str(f)]
if FOLDS_PATH is None:
    FOLDS_PATH = SOURCE_CV_DIR / 'folds.csv'

if not ((DATA_DIR / 'train.csv').is_file() and (DATA_DIR / 'test.csv').is_file()):
    candidates = []
    for directory, subdirs, files in os.walk('/kaggle/input'):
        subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
        root = Path(directory)
        if {'train.csv', 'test.csv'}.issubset(files) and (root / 'train').is_dir() and (root / 'test').is_dir():
            if {'image_id', *LABEL_COLS}.issubset(pd.read_csv(root / 'train.csv', nrows=0).columns):
                candidates.append(root)
    if len(candidates) != 1:
        raise FileNotFoundError(f'Set DATA_DIR to the competition data. Candidates: {candidates}')
    DATA_DIR = candidates[0]
if FOLDS_PATH is None:
    paths = [Path('/kaggle/working/folds.csv'), Path('folds.csv'), CV_DIR / 'folds.csv']
    if PREVIOUS_CV_DIR is not None:
        paths.append(Path(PREVIOUS_CV_DIR) / 'folds.csv')
    candidates = {p.resolve() for p in paths if p.is_file()}
    if not candidates:
        for directory, subdirs, files in os.walk('/kaggle/input'):
            subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
            if 'folds.csv' in files:
                candidates.add((Path(directory) / 'folds.csv').resolve())
    # Copies with identical contents represent the same split.
    by_contents = {}
    for path in sorted(candidates):
        by_contents.setdefault(file_digest(path), path)
    if len(by_contents) != 1:
        raise FileNotFoundError(f'Attach the ORIGINAL folds.csv or set FOLDS_PATH. No split is generated. Candidates: {sorted(candidates)}')
    FOLDS_PATH = next(iter(by_contents.values()))
FOLDS_PATH = Path(FOLDS_PATH)
train = pd.read_csv(DATA_DIR / 'train.csv', dtype={'image_id': str})
folds = pd.read_csv(FOLDS_PATH, dtype={'image_id': str})
assert {'image_id', 'cluster', 'fold', *LABEL_COLS}.issubset(folds.columns)
assert train.image_id.notna().all() and train.image_id.is_unique
assert folds.image_id.notna().all() and folds.image_id.is_unique
assert set(train.image_id) == set(folds.image_id)
assert train[LABEL_COLS].isin([0, 1]).all().all()
aligned = folds.set_index('image_id').loc[train.image_id]
assert np.array_equal(train[LABEL_COLS].to_numpy(), aligned[LABEL_COLS].to_numpy())
assert folds[['cluster', 'fold']].notna().all().all()
assert set(folds.fold.unique()) == set(range(5))
assert folds.groupby('cluster').fold.nunique().eq(1).all(), 'A group crosses folds.'
frame = train.merge(folds[['image_id', 'cluster', 'fold']], on='image_id', validate='one_to_one', sort=False)
frame['path'] = frame.image_id.map(lambda ident: str(DATA_DIR / 'train' / f'{ident}.jpg'))
canonical = folds[['image_id', *LABEL_COLS, 'cluster', 'fold']].sort_values('image_id').copy()
canonical[['cluster', 'fold', *LABEL_COLS]] = canonical[['cluster', 'fold', *LABEL_COLS]].astype('int64')
folds_fingerprint = digest_text(canonical.to_csv(index=False, lineterminator=chr(10)))
assert folds_fingerprint == source_manifest['folds_fingerprint'], 'Source and target splits differ.'
cv_config = {
    'pipeline': 'convnext_cv_320_fast_ema_v1', 'model_name': MODEL_NAME, 'label_cols': LABEL_COLS,
    'versions': {'torch': str(torch.__version__), 'timm': str(timm.__version__), 'albumentations': str(A.__version__)},
    'source_cv_fingerprint': source_manifest['cv_fingerprint'],
    'source_checkpoint_hashes': source_manifest['checkpoint_hashes'],
    'head_lr': HEAD_LR, 'accumulation_steps': ACCUMULATION_STEPS,
    'ema_decay': EMA_DECAY, 'warmup_epochs': WARMUP_EPOCHS,
    'epoch_validation': 'ema_only', 'tta': False, 'channels_last': CHANNELS_LAST,
    'train_budget_minutes': TRAIN_BUDGET_MINUTES,
    'scheduler': 'per_optimizer_step_warmup_cosine',
    'folds_fingerprint': folds_fingerprint, 'seed': SEED, 'epochs': EPOCHS,
    'batch_size': BATCH_SIZE, 'lr': LR, 'min_lr': MIN_LR, 'weight_decay': WEIGHT_DECAY,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE, 'preprocessing': preprocessing,
    'train_loss': 'BCE_sqrt_pos_weight', 'val_loss': 'BCE_unweighted', 'threshold': THRESHOLD,
}
cv_fingerprint = digest_text(json.dumps(cv_config, sort_keys=True))
cv_config['cv_fingerprint'] = cv_fingerprint
config_path = CV_DIR / 'cv_config.json'
if config_path.is_file():
    assert json.loads(config_path.read_text()) == cv_config, 'CV settings changed; use a new CV_DIR.'
if PREVIOUS_CV_DIR is not None:
    PREVIOUS_CV_DIR = Path(PREVIOUS_CV_DIR)
    previous_config = json.loads((PREVIOUS_CV_DIR / 'cv_config.json').read_text())
    assert previous_config == cv_config, 'Previous CV output uses different settings or folds.'
    # Carry all saved folds forward even when training only one requested fold now.
    for previous_f in range(5):
        source_folder = PREVIOUS_CV_DIR / f'fold_{previous_f}'
        destination = CV_DIR / f'fold_{previous_f}'
        if not any((destination / name).exists() for name in ['best.pt', 'last.pt', 'complete.json']):
            destination.mkdir(parents=True, exist_ok=True)
            for name in ['best.pt', 'last.pt', 'history.csv', 'oof.csv', 'complete.json', 'wandb_id.txt', 'comparison.json']:
                if (source_folder / name).is_file():
                    shutil.copy2(source_folder / name, destination / name)

save_json(cv_config, config_path)
folds.to_csv(CV_DIR / 'folds.csv', index=False)
print('Original folds:', FOLDS_PATH, '| fingerprint:', folds_fingerprint[:12])
display(frame.groupby('fold')[LABEL_COLS].sum())

# Validate cached 224 test predictions BEFORE spending GPU time.
def compatible_source_predictions(directory):
    manifest_path = directory / 'ensemble_config.json'
    if not manifest_path.is_file() or not (directory / 'test_probabilities_original.csv').is_file():
        return False
    manifest = json.loads(manifest_path.read_text())
    return (manifest.get('cv_fingerprint') == source_manifest['cv_fingerprint']
            and manifest.get('checkpoint_hashes') == source_manifest['checkpoint_hashes'])

if not compatible_source_predictions(SOURCE_PREDICTIONS_DIR):
    candidates = []
    for directory, subdirs, files in os.walk('/kaggle/input'):
        subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
        if compatible_source_predictions(Path(directory)):
            candidates.append(Path(directory))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Attach convnext_cv_ensemble with its ORIGINAL test probabilities, or set SOURCE_PREDICTIONS_DIR: {candidates}')
    SOURCE_PREDICTIONS_DIR = candidates[0]
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'image_id': str})
test = pd.read_csv(DATA_DIR / 'test.csv', dtype={'image_id': str})
assert sample.image_id.notna().all() and sample.image_id.is_unique
assert test.image_id.notna().all() and test.image_id.is_unique
assert set(sample.columns) == {'image_id', *LABEL_COLS} and set(sample.image_id) == set(test.image_id)
source_test = pd.read_csv(SOURCE_PREDICTIONS_DIR / 'test_probabilities_original.csv', dtype={'image_id': str})
assert source_test.image_id.notna().all() and source_test.image_id.is_unique
assert set(source_test.image_id) == set(sample.image_id)
source_test = source_test.set_index('image_id').loc[sample.image_id].reset_index()
source_test_p = source_test[LABEL_COLS].to_numpy(dtype=np.float64)
assert np.isfinite(source_test_p).all() and ((source_test_p >= 0) & (source_test_p <= 1)).all()
print('224 test inference is cached:', SOURCE_PREDICTIONS_DIR)


Original folds: /kaggle/input/datasets/nikron62/convnext-backup/convnext_cv/folds.csv | fingerprint: 453ed1d2e114


,Cat,Dog,Moris,Motya,Biatrix
fold,,,,,
0,1006,982,14,7,9
1,1027,1007,12,7,7
2,1024,1002,12,14,5
3,1031,1003,7,4,14
4,989,1031,11,10,6


224 test inference is cached: /kaggle/input/datasets/nikron62/convnext-backup/convnext_cv_ensemble


## Source OOF diagnostics (no training)

Report precision, recall, FP/FN and F1 for each label using the saved 224 thresholds.
Save the error rows for inspection; do not manually relabel competition images.
Rare-label counts are small, so a few mistakes can move macro F1 substantially.


In [ ]:
source_oof = pd.read_csv(SOURCE_CV_DIR / 'oof_predictions.csv', dtype={'image_id': str})
assert source_oof.image_id.is_unique and set(source_oof.image_id) == set(frame.image_id)
source_oof = source_oof.set_index('image_id').loc[frame.image_id].reset_index()
assert np.array_equal(source_oof[LABEL_COLS].to_numpy(), frame[LABEL_COLS].to_numpy())
assert np.array_equal(source_oof['fold'].to_numpy(), frame['fold'].to_numpy())
source_targets = source_oof[LABEL_COLS].to_numpy(dtype=np.int32)
audit_rows, errors = [], []
for method, suffix in [('original', '_prob')]:
    probabilities = source_oof[[label + suffix for label in LABEL_COLS]].to_numpy()
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    thresholds = np.asarray(source_manifest['thresholds'][method], dtype=np.float64)
    assert thresholds.shape == (len(LABEL_COLS),) and np.isfinite(thresholds).all()
    assert ((thresholds > 0) & (thresholds < 1)).all()
    for j, label in enumerate(LABEL_COLS):
        y = source_targets[:, j]
        predicted = probabilities[:, j] >= thresholds[j]
        fp, fn = predicted & (y == 0), (~predicted) & (y == 1)
        audit_rows.append({'method': method, 'label': label, 'positives': int(y.sum()),
                           'threshold': float(thresholds[j]), 'false_positives': int(fp.sum()),
                           'false_negatives': int(fn.sum()),
                           'precision': precision_score(y, predicted, zero_division=0),
                           'recall': recall_score(y, predicted, zero_division=0),
                           'f1': f1_score(y, predicted, zero_division=0)})
        for index in np.flatnonzero(fp | fn):
            errors.append({'image_id': source_oof.image_id.iloc[index], 'fold': int(source_oof.fold.iloc[index]),
                           'method': method, 'label': label, 'target': int(y[index]),
                           'probability': float(probabilities[index, j]), 'threshold': float(thresholds[j]),
                           'error': 'FP' if fp[index] else 'FN'})
audit = pd.DataFrame(audit_rows)
display(audit)
audit.to_csv(CV_DIR / 'source_224_per_label.csv', index=False)
pd.DataFrame(errors, columns=['image_id', 'fold', 'method', 'label', 'target', 'probability', 'threshold', 'error']).to_csv(CV_DIR / 'source_224_errors.csv', index=False)

,method,label,positives,threshold,false_positives,false_negatives,precision,recall,f1
0,original,Cat,5077,0.39,23,29,0.995464,0.994288,0.994876
1,original,Dog,5025,0.39,35,34,0.993036,0.993234,0.993135
2,original,Moris,56,0.62,1,3,0.981481,0.946429,0.963636
3,original,Motya,42,0.50,2,1,0.953488,0.976190,0.964706
4,original,Biatrix,41,0.50,0,2,1.000000,0.951220,0.975000


## 3. Training helpers
The original transforms, dataset and validation metrics are reused. AMP and channels-last
are used on CUDA. Validation runs once per epoch, using EMA weights only.

In [ ]:
def make_transforms():
    train_transform = A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=preprocessing['interpolation']),
        A.HorizontalFlip(p=0.5),
        A.Affine(translate_percent=(-0.1, 0.1), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
        A.Normalize(mean=preprocessing['mean'], std=preprocessing['std']),
        ToTensorV2(),
    ])
    eval_transform = A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=preprocessing['interpolation']),
        A.Normalize(mean=preprocessing['mean'], std=preprocessing['std']),
        ToTensorV2(),
    ])
    return train_transform, eval_transform

class PetDataset(Dataset):
    def __init__(self, frame, transform, with_labels=True):
        self.paths = frame.path.tolist()
        self.targets = frame[LABEL_COLS].to_numpy(dtype=np.float32) if with_labels else None
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            pixels = np.array(image.convert('RGB'))
        x = self.transform(image=pixels)['image']
        if self.targets is None:
            return x
        return x, torch.from_numpy(self.targets[index])

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(frame, training=False, with_labels=True):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        PetDataset(frame, train_transform if training else eval_transform, with_labels),
        batch_size=BATCH_SIZE, shuffle=training, num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda', drop_last=False,
        worker_init_fn=seed_worker, generator=generator,
    )

In [ ]:
def metric_report(targets, probabilities, threshold=THRESHOLD):
    predictions = (probabilities >= threshold).astype(np.int32)
    rows = []
    for j, label in enumerate(LABEL_COLS):
        y = targets[:, j]
        auc = roc_auc_score(y, probabilities[:, j]) if np.unique(y).size == 2 else np.nan
        rows.append({
            'label': label,
            'positives': int(y.sum()),
            'predicted positives': int(predictions[:, j].sum()),
            'f1': f1_score(y, predictions[:, j], zero_division=0),
            'precision': precision_score(y, predictions[:, j], zero_division=0),
            'recall': recall_score(y, predictions[:, j], zero_division=0),
            'roc_auc': auc,
        })
    table = pd.DataFrame(rows).set_index('label')
    valid_auc = table.roc_auc.dropna()
    metrics = {
        'macro_f1': float(f1_score(targets, predictions, average='macro', zero_division=0)),
        'macro_roc_auc': float(valid_auc.mean()) if len(valid_auc) else float('nan'),
    }
    return metrics, table

def tune_thresholds(targets, probabilities):
    thresholds = np.full(len(LABEL_COLS), THRESHOLD, dtype=np.float64)
    candidates = np.round(np.linspace(0.05, 0.95, 91), 2)
    for j, label in enumerate(LABEL_COLS):
        if np.unique(targets[:, j]).size < 2:
            print(f'{label}: keeping {THRESHOLD}; validation does not contain both outcomes.')
            continue
        scores = np.array([
            f1_score(targets[:, j], probabilities[:, j] >= t, zero_division=0)
            for t in candidates
        ])
        tied = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-12))
        best = tied[np.argmin(np.abs(candidates[tied] - THRESHOLD))]
        thresholds[j] = candidates[best]
    return thresholds

In [ ]:

@torch.no_grad()
def update_ema():
    live = model.state_dict()
    for name, averaged in ema_model.state_dict().items():
        value = live[name].detach()
        if averaged.is_floating_point():
            averaged.mul_(EMA_DECAY).add_(value, alpha=1 - EMA_DECAY)
        else:
            averaged.copy_(value)

def train_one_epoch():
    global train_step
    model.train()
    total_loss, seen = 0.0, 0
    progress = tqdm(train_loader, desc='fine-tune', leave=False)
    for batch_index, (images, targets) in enumerate(progress):
        if batch_index % ACCUMULATION_STEPS == 0:
            optimizer.zero_grad(set_to_none=True)
            window_examples = min(BATCH_SIZE * ACCUMULATION_STEPS, len(train_loader.dataset) - seen)
        images = images.to(DEVICE, non_blocking=True, memory_format=MEMORY_FORMAT)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            loss = train_criterion(model(images), targets)
        if not torch.isfinite(loss):
            raise FloatingPointError(f'Non-finite training loss: {loss.item()}')
        scaler.scale(loss * len(images) / window_examples).backward()
        total_loss += loss.item() * len(images)
        seen += len(images)
        end_window = (batch_index + 1) % ACCUMULATION_STEPS == 0 or batch_index + 1 == len(train_loader)
        if end_window:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                update_ema()
                scheduler.step()
                train_step += 1
            if wandb_run is not None and (train_step % WANDB_LOG_EVERY == 0 or batch_index + 1 == len(train_loader)):
                wandb_run.log({'train_step': train_step, 'batch/loss': loss.item(),
                               'batch/running_loss': total_loss / seen})
        progress.set_postfix(loss=f'{total_loss / seen:.5f}')
    return total_loss / seen

@torch.inference_mode()
def validate():
    model.eval()
    total_loss = 0.0
    all_targets, all_probabilities = [], []
    for images, targets in tqdm(val_loader, desc='validation', leave=False):
        images = images.to(DEVICE, non_blocking=True, memory_format=MEMORY_FORMAT)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
        loss = val_criterion(logits.float(), targets)
        total_loss += loss.item() * len(images)
        all_targets.append(targets.cpu().numpy())
        all_probabilities.append(logits.float().sigmoid().cpu().numpy())
    targets = np.concatenate(all_targets)
    probabilities = np.concatenate(all_probabilities)
    assert np.isfinite(probabilities).all(), 'Non-finite validation probabilities.'
    metrics, table = metric_report(targets, probabilities)
    metrics['val_loss'] = total_loss / len(val_df)
    assert np.isfinite(metrics['val_loss']), 'Non-finite validation loss.'
    return metrics, table, probabilities

In [ ]:
wandb_run = None
train_step = 0

def start_wandb_run(config):
    if not USE_WANDB or WANDB_MODE == 'disabled':
        return nullcontext(None)
    if WANDB_MODE not in {'online', 'offline'}:
        raise ValueError('WANDB_MODE must be online, offline, or disabled.')
    if WANDB_LOG_EVERY < 1:
        raise ValueError('WANDB_LOG_EVERY must be positive.')
    try:
        import wandb
    except ImportError:
        raise ImportError('Install wandb with %pip install -q wandb, or set USE_WANDB = False.') from None

    if WANDB_MODE == 'online' and not os.environ.get('WANDB_API_KEY'):
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
        except Exception:
            raise RuntimeError(
                'Add the WANDB_API_KEY Kaggle Secret and grant notebook access, '
                'or set the WANDB_API_KEY environment variable. '
                'For local logging use WANDB_MODE = "offline".'
            ) from None
        if not api_key:
            raise RuntimeError('WANDB_API_KEY is empty. Check the Kaggle Secret.')
        os.environ['WANDB_API_KEY'] = api_key  # never print or put the key in run config

    id_path = OUTPUT_DIR / 'wandb_id.txt'
    run_id = id_path.read_text().strip() if id_path.is_file() else wandb.util.generate_id()
    id_path.write_text(run_id)
    return wandb.init(
        id=run_id, resume='allow', group=f'convnext-cv320-fast-{cv_fingerprint[:8]}',
        project=WANDB_PROJECT, entity=WANDB_ENTITY, name=WANDB_RUN_NAME,
        mode=WANDB_MODE, config=config, dir=str(OUTPUT_DIR), save_code=False,
    )

In [ ]:
def check_oof(path, held_out):
    result = pd.read_csv(path, dtype={'image_id': str})
    assert result.image_id.is_unique and set(result.image_id) == set(held_out.image_id)
    result = result.set_index('image_id').loc[held_out.image_id].reset_index()
    assert np.array_equal(result[LABEL_COLS].to_numpy(), held_out[LABEL_COLS].to_numpy())
    assert np.array_equal(result['fold'].to_numpy(), held_out['fold'].to_numpy())
    assert np.array_equal(result['cluster'].to_numpy(), held_out['cluster'].to_numpy())
    for suffix in ('_prob',):
        p = result[[label + suffix for label in LABEL_COLS]].to_numpy()
        assert p.shape == (len(held_out), len(LABEL_COLS)) and np.isfinite(p).all()
        assert ((p >= 0) & (p <= 1)).all()
    return result

def create_inference_backup():
    # Exclude optimizer/resume states and W&B logs to keep the download near the
    # size of five best models. This is a local snapshot, not persistent storage.
    from zipfile import ZipFile, ZIP_STORED
    roots = [CV_DIR, TEST_OUTPUT_DIR, BLEND_OUTPUT_DIR]
    paths = sorted(path for root in roots if root.is_dir() for path in root.rglob('*')
                   if path.is_file() and 'wandb' not in path.relative_to(root).parts
                   and path.name != 'last.pt' and '.tmp' not in path.name)
    temporary = BACKUP_PATH.with_suffix('.tmp.zip')
    with ZipFile(temporary, 'w', compression=ZIP_STORED) as archive:
        for path in paths:
            archive.write(path, arcname=path.relative_to(CV_DIR.parent).as_posix())
    temporary.replace(BACKUP_PATH)
    print(f'Backup refreshed: {BACKUP_PATH} ({BACKUP_PATH.stat().st_size / 1024**3:.2f} GiB). Download it before ending the session.')


## 4. Same-fold fine-tuning with EMA and a soft time allowance
Every unfinished fold receives at least one epoch. The remaining training allowance
is shared across unfinished folds; a fold stops at an epoch boundary when another
epoch is unlikely to fit its share. Maximum four epochs, patience two. Only EMA is
evaluated after each epoch. Validation probabilities are stored in best.pt together
with their exact IDs and reused for OOF, avoiding another inference pass.

In [ ]:
for FOLD in FOLDS_TO_TRAIN:
    OUTPUT_DIR = CV_DIR / f'fold_{FOLD}'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    BEST_PATH, LAST_PATH = OUTPUT_DIR / 'best.pt', OUTPUT_DIR / 'last.pt'
    complete_path, oof_path = OUTPUT_DIR / 'complete.json', OUTPUT_DIR / 'oof.csv'
    train_df = frame.loc[frame.fold != FOLD].reset_index(drop=True)
    val_df = frame.loc[frame.fold == FOLD].reset_index(drop=True)
    assert len(train_df) and len(val_df) and set(train_df.cluster).isdisjoint(val_df.cluster)
    if complete_path.is_file():
        done = json.loads(complete_path.read_text())
        assert done['cv_fingerprint'] == cv_fingerprint and done['fold'] == FOLD
        assert done['checkpoint_sha256'] == file_digest(BEST_PATH)
        check_oof(oof_path, val_df)
        print(f'Fold {FOLD}: completed; skipping.')
        continue

    remaining_folds = sum(not (CV_DIR / f'fold_{f}' / 'complete.json').is_file() for f in FOLDS_TO_TRAIN)
    remaining_seconds = max(0.0, TRAIN_DEADLINE - perf_counter())
    fold_deadline = perf_counter() + remaining_seconds / max(remaining_folds, 1)
    print(f'Fold {FOLD}: soft allowance {remaining_seconds / max(remaining_folds, 1) / 60:.1f} min; at most {EPOCHS} epochs.')
    source_path = SOURCE_CV_DIR / f'fold_{FOLD}' / 'best.pt' 
    assert file_digest(source_path) == source_manifest['checkpoint_hashes'][str(FOLD)]
    # Prove that the matching source model held out the same images.
    check_oof(SOURCE_CV_DIR / f'fold_{FOLD}' / 'oof.csv', val_df)
    source_checkpoint = torch.load(source_path, map_location='cpu', weights_only=True)
    cfg = source_checkpoint['config']
    assert cfg['fold'] == FOLD and cfg['cv_fingerprint'] == source_manifest['cv_fingerprint']
    assert cfg['model_name'] == MODEL_NAME and cfg['label_cols'] == LABEL_COLS
    assert cfg['preprocessing'] == source_manifest['preprocessing']
    seed_everything(SEED + FOLD)
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(LABEL_COLS)).to(device=DEVICE, memory_format=MEMORY_FORMAT)
    model.load_state_dict(source_checkpoint['model'], strict=True)
    source_epoch = source_checkpoint['epoch']
    del source_checkpoint
    ema_model = copy.deepcopy(model).eval().requires_grad_(False)
    train_transform, eval_transform = make_transforms()
    train_loader, val_loader = make_loader(train_df, training=True), make_loader(val_df)
    positive_counts = train_df[LABEL_COLS].sum().to_numpy(dtype=np.float32)
    negative_counts = len(train_df) - positive_counts
    assert (positive_counts > 0).all() and (negative_counts > 0).all()
    pos_weight = torch.as_tensor(np.sqrt(negative_counts / positive_counts), device=DEVICE)
    train_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    val_criterion = nn.BCEWithLogitsLoss()
    classifier = list(model.get_classifier().parameters())
    head_ids = {id(parameter) for parameter in classifier}
    backbone = [parameter for parameter in model.parameters() if id(parameter) not in head_ids]
    assert classifier and backbone
    optimizer = torch.optim.AdamW([{'params': backbone, 'lr': LR}, {'params': classifier, 'lr': HEAD_LR}], weight_decay=WEIGHT_DECAY)
    updates_per_epoch = math.ceil(len(train_loader) / ACCUMULATION_STEPS)
    total_updates = EPOCHS * updates_per_epoch
    warmup_updates = WARMUP_EPOCHS * updates_per_epoch
    def lr_factor(update):
        if update < warmup_updates:
            return 0.1 + 0.9 * update / max(warmup_updates, 1)
        progress = min(1.0, (update - warmup_updates) / max(total_updates - warmup_updates, 1))
        minimum = MIN_LR / LR
        return minimum + 0.5 * (1 - minimum) * (1 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_factor)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    data_config = timm.data.resolve_data_config({}, model=model)
    data_config['input_size'] = (3, IMAGE_SIZE, IMAGE_SIZE)
    fold_config = {**cv_config, 'fold': FOLD, 'num_classes': len(LABEL_COLS),
                   'image_size': IMAGE_SIZE, 'folds_path': str(FOLDS_PATH), 'data_config': data_config,
                   'source_checkpoint': str(source_path), 'source_epoch': source_epoch,
                   'pos_weight': pos_weight.cpu().tolist()}
    WANDB_RUN_NAME = f'convnext-cv320-fast-fold{FOLD}'
    wandb_run, train_step = None, 0
    history, start_epoch, patience, training_complete = [], 1, 0, False
    fresh = not LAST_PATH.is_file()
    if not fresh:
        last = torch.load(LAST_PATH, map_location='cpu', weights_only=True)
        assert last['config']['cv_fingerprint'] == cv_fingerprint and last['config']['fold'] == FOLD
        assert BEST_PATH.is_file()
        model.load_state_dict(last['model'])
        ema_model.load_state_dict(last['ema'])
        optimizer.load_state_dict(last['optimizer'])
        scheduler.load_state_dict(last['scheduler'])
        scaler.load_state_dict(last['scaler'])
        history, start_epoch = last['history'], last['epoch'] + 1
        best_f1, best_loss, patience = last['best_f1'], last['best_loss'], last['patience']
        train_step, training_complete = last['train_step'], last['training_complete']
        initial_metrics = last['initial_metrics']
        print(f'Fold {FOLD}: resuming after fine-tuning epoch {last["epoch"]}.')
        del last
    else:
        if BEST_PATH.is_file():
            orphan = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
            assert orphan['epoch'] == 0 and orphan['config']['cv_fingerprint'] == cv_fingerprint, 'Restore last.pt or use a fresh CV_DIR.'
            del orphan
        initial_metrics, _, initial_probabilities = validate()
        val_targets = val_df[LABEL_COLS].to_numpy(dtype=np.int32)
        initial_metrics['val_accuracy'] = float(((initial_probabilities >= THRESHOLD) == val_targets).all(axis=1).mean())
        best_f1, best_loss = initial_metrics['macro_f1'], initial_metrics['val_loss']
        save_torch({'model': model.state_dict(), 'epoch': 0, 'metrics': initial_metrics,
                    'variant': 'source_at_320', 'config': fold_config,
                    'validation_probabilities': torch.from_numpy(initial_probabilities.copy()),
                    'validation_image_ids': val_df.image_id.tolist()}, BEST_PATH)
        # Initial last.pt permits recovery even if the first epoch is interrupted.
        save_torch({'model': model.state_dict(), 'ema': ema_model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(), 'epoch': 0,
                    'history': history, 'best_f1': best_f1, 'best_loss': best_loss, 'patience': patience,
                    'train_step': train_step, 'training_complete': False, 'initial_metrics': initial_metrics,
                    'config': fold_config}, LAST_PATH)
        print(f'Fold {FOLD}: source weights at 320, F1={best_f1:.5f} before fine-tuning.')
    try:
        if not training_complete:
            with start_wandb_run(fold_config) as wandb_run:
                if wandb_run is not None:
                    wandb_run.define_metric('train_step')
                    wandb_run.define_metric('batch/*', step_metric='train_step')
                    wandb_run.define_metric('epoch')
                    wandb_run.define_metric('epoch_metrics/*', step_metric='epoch')
                    for metric in ('val_f1', 'val_loss', 'train_loss_epoch', 'lr'):
                        wandb_run.define_metric(metric, step_metric='epoch')
                    print('W&B:', wandb_run.url if WANDB_MODE == 'online' else wandb_run.dir)
                    if fresh:
                        wandb_run.log({'epoch': 0, 'val_f1': initial_metrics['macro_f1'],
                                       'val_loss': initial_metrics['val_loss'],
                                       'epoch_metrics/val_f1': initial_metrics['macro_f1']})
                for epoch in range(start_epoch, EPOCHS + 1):
                    epoch_seed = SEED + FOLD * 1000 + epoch
                    seed_everything(epoch_seed)
                    if hasattr(train_transform, 'set_random_seed'):
                        train_transform.set_random_seed(epoch_seed)
                    train_loader.generator.manual_seed(epoch_seed)
                    started = perf_counter()
                    epoch_lr, head_lr = [group['lr'] for group in optimizer.param_groups]
                    train_loss = train_one_epoch()
                    raw_model = model
                    try:
                        model = ema_model
                        metrics, table, probabilities = validate()
                    finally:
                        model = raw_model
                    chosen, variant = ema_model, 'ema'
                    targets = val_df[LABEL_COLS].to_numpy(dtype=np.int32)
                    metrics['val_accuracy'] = float(((probabilities >= THRESHOLD) == targets).all(axis=1).mean())
                    improved = metrics['macro_f1'] > best_f1
                    patience = 0 if improved else patience + 1
                    if improved or (metrics['macro_f1'] == best_f1 and metrics['val_loss'] < best_loss):
                        best_f1, best_loss = metrics['macro_f1'], metrics['val_loss']
                        save_torch({'model': chosen.state_dict(), 'epoch': epoch, 'metrics': metrics,
                                    'variant': variant, 'config': fold_config,
                                    'validation_probabilities': torch.from_numpy(probabilities.copy()),
                                    'validation_image_ids': val_df.image_id.tolist()}, BEST_PATH)
                    row = {'epoch': epoch, 'train_loss': train_loss, **metrics, 'lr': epoch_lr, 'head_lr': head_lr,
                           'val_f1_ema': metrics['macro_f1'],
                           'epoch_seconds': perf_counter() - started, 'epochs_without_improvement': patience}
                    history.append(row)
                    estimated_next_epoch = max(item['epoch_seconds'] for item in history[-2:])
                    budget_stop = perf_counter() + estimated_next_epoch > fold_deadline
                    training_complete = epoch == EPOCHS or patience >= EARLY_STOPPING_PATIENCE or budget_stop
                    if budget_stop:
                        print('Stopping this fold at the epoch boundary to leave time for the remaining folds and inference.')
                    save_torch({'model': model.state_dict(), 'ema': ema_model.state_dict(), 'optimizer': optimizer.state_dict(),
                                'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(), 'epoch': epoch,
                                'history': history, 'best_f1': best_f1, 'best_loss': best_loss, 'patience': patience,
                                'train_step': train_step, 'training_complete': training_complete,
                                'initial_metrics': initial_metrics, 'config': fold_config}, LAST_PATH)
                    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'history.csv', index=False)
                    print(f'Fold {FOLD} | epoch {epoch}/{EPOCHS} | EMA F1 {metrics["macro_f1"]:.5f} | best {best_f1:.5f} | epoch {row["epoch_seconds"]:.0f}s | patience {patience}/{EARLY_STOPPING_PATIENCE}')
                    if wandb_run is not None:
                        logs = {'epoch': epoch, 'val_f1': metrics['macro_f1'], 'val_loss': metrics['val_loss'],
                                'train_loss_epoch': train_loss, 'lr': epoch_lr}
                        logs.update({f'epoch_metrics/{key}': value for key, value in row.items() if key != 'epoch' and np.isfinite(value)})
                        logs['epoch_metrics/val_f1'] = metrics['macro_f1']
                        logs.update({f'epoch_metrics/f1_{label}': float(table.loc[label, 'f1']) for label in LABEL_COLS})
                        wandb_run.log(logs)
                        wandb_run.summary.update({'best/macro_f1': best_f1, 'completed_epochs': epoch})
                    if training_complete:
                        break
    finally:
        wandb_run = None
    best_checkpoint = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
    assert best_checkpoint['validation_image_ids'] == val_df.image_id.tolist()
    originals = best_checkpoint['validation_probabilities'].cpu().numpy()
    result = val_df[['image_id', 'cluster', 'fold', *LABEL_COLS]].copy()
    for j, label in enumerate(LABEL_COLS):
        result[label + '_prob'] = originals[:, j]
    result.to_csv(oof_path, index=False)
    check_oof(oof_path, val_df)
    source_rows = source_oof.set_index('image_id').loc[val_df.image_id]
    source_p = source_rows[[label + '_prob' for label in LABEL_COLS]].to_numpy()
    source_224_f1 = f1_score(val_df[LABEL_COLS].to_numpy(), source_p >= THRESHOLD, average='macro', zero_division=0)
    comparison = {'fold': FOLD, 'source_224_f1_at_0_5': float(source_224_f1),
                  'source_at_320_f1_at_0_5': initial_metrics['macro_f1'],
                  'best_320_f1_at_0_5': best_checkpoint['metrics']['macro_f1'],
                  'best_epoch': best_checkpoint['epoch'], 'best_variant': best_checkpoint['variant']}
    save_json(comparison, OUTPUT_DIR / 'comparison.json')
    save_json({'fold': FOLD, 'cv_fingerprint': cv_fingerprint, 'checkpoint_sha256': file_digest(BEST_PATH),
               'best_epoch': best_checkpoint['epoch'], 'validation_images': len(val_df)}, complete_path)
    print('Completed:', comparison)
    create_inference_backup()
    for name in ('model', 'ema_model', 'raw_model', 'chosen', 'classifier', 'backbone', 'optimizer', 'scheduler', 'scaler', 'best_checkpoint'):
        globals().pop(name, None)
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

Fold 0: soft allowance 19.0 min; at most 4 epochs.


Fold 0: source weights at 320, F1=0.92439 before fine-tuning.


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mkovalenko (mkovalenko-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B: https://wandb.ai/yrachynska-kyiv-school-of-economics/cats-and-dogs/runs/fsv6524b


Fold 0 | epoch 1/4 | EMA F1 0.95924 | best 0.95924 | epoch 341s | patience 0/2


...

Fold 4 | epoch 2/4 | EMA F1 0.95337 | best 0.95337 | epoch 171s | patience 0/2


Fold 4 | epoch 3/4 | EMA F1 0.95317 | best 0.95337 | epoch 169s | patience 1/2


Fold 4 | epoch 4/4 | EMA F1 0.96202 | best 0.96202 | epoch 169s | patience 0/2


batch/loss,▂▁▆▁▃▃▁▂▁▂▁▁▁▂▁▁█▁▁▂▁▁▁▁▂▁▁▁▁▁▁▁▁
batch/running_loss,█▆▅▅▄▄▄▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▃▅▆█
epoch_metrics/epoch_seconds,█▂▁▁
epoch_metrics/epochs_without_improvement,▁▁█▁
epoch_metrics/f1_Biatrix,▁▁▁▁
epoch_metrics/f1_Cat,██▁▁
epoch_metrics/f1_Dog,▁▆▃█
epoch_metrics/f1_Moris,▁▁▁█
epoch_metrics/f1_Motya,▁███
+14,...


Completed: {'fold': 4, 'source_224_f1_at_0_5': 0.9670232124992101, 'source_at_320_f1_at_0_5': 0.9401950413097797, 'best_320_f1_at_0_5': 0.9620245706218118, 'best_epoch': 4, 'best_variant': 'ema'}
Backup refreshed: /kaggle/working/convnext_320_fast_inference_backup.zip (0.52 GiB). Download it before ending the session.


## 5. Full OOF calibration, without TTA
Require all five held-out folds and verify their checkpoint hashes before fitting
thresholds. Fitted OOF F1 is a calibration score, not an independent test estimate.

In [ ]:
missing_folds = [f for f in range(5) if not (CV_DIR / f'fold_{f}' / 'complete.json').is_file()]
if missing_folds:
    print('OOF calibration not ready. Train/import remaining folds:', missing_folds)
else:
    parts = []
    checkpoint_hashes = {}
    for f in range(5):
        folder = CV_DIR / f'fold_{f}'
        done = json.loads((folder / 'complete.json').read_text())
        assert done['cv_fingerprint'] == cv_fingerprint and done['fold'] == f
        assert done['checkpoint_sha256'] == file_digest(folder / 'best.pt')
        checkpoint_hashes[str(f)] = done['checkpoint_sha256']
        held_out = frame.loc[frame.fold == f].reset_index(drop=True)
        parts.append(check_oof(folder / 'oof.csv', held_out))
    oof = pd.concat(parts, ignore_index=True)
    assert oof.image_id.is_unique and set(oof.image_id) == set(frame.image_id)
    oof = oof.set_index('image_id').loc[frame.image_id].reset_index()
    targets = frame[LABEL_COLS].to_numpy(dtype=np.int32)
    all_thresholds, reports = {}, []
    for method, suffix in [('original', '_prob')]:
        probabilities = oof[[label + suffix for label in LABEL_COLS]].to_numpy()
        thresholds = tune_thresholds(targets, probabilities)
        all_thresholds[method] = thresholds.tolist()
        for label, cutoffs in [('fixed_0_5', 0.5), ('fitted_oof', thresholds)]:
            report, _ = metric_report(targets, probabilities, cutoffs)
            report['val_accuracy'] = float(((probabilities >= cutoffs) == targets).all(axis=1).mean())
            reports.append({'method': method, 'threshold_source': label, **report})
    summary = pd.DataFrame(reports)
    display(summary)
    oof.to_csv(CV_DIR / 'oof_predictions.csv', index=False)
    summary.to_csv(CV_DIR / 'oof_metrics.csv', index=False)
    save_json({'model_name': MODEL_NAME, 'label_cols': LABEL_COLS, 'folds': list(range(5)),
               'cv_fingerprint': cv_fingerprint, 'folds_fingerprint': folds_fingerprint,
               'preprocessing': preprocessing, 'thresholds': all_thresholds,
               'checkpoint_hashes': checkpoint_hashes,
               'aggregation': 'equal_mean_sigmoid_probabilities', 'tta': False,
               'threshold_source': 'all_held_out_oof_predictions'}, CV_DIR / 'ensemble_config.json')
    print('OOF ready. Continue below: original-only test inference and 224/320 blend.')

comparisons = [json.loads((CV_DIR / f'fold_{f}' / 'comparison.json').read_text())
               for f in range(5) if (CV_DIR / f'fold_{f}' / 'comparison.json').is_file()]
if comparisons:
    display(pd.DataFrame(comparisons))


,method,threshold_source,macro_f1,macro_roc_auc,val_accuracy
0,original,fixed_0_5,0.976476,0.999757,0.992822
1,original,fitted_oof,0.978790,0.999757,0.992985


OOF ready. Continue below: original-only test inference and 224/320 blend.


,fold,source_224_f1_at_0_5,source_at_320_f1_at_0_5,best_320_f1_at_0_5,best_epoch,best_variant
0,0,0.972387,0.924389,0.972481,4,ema
1,1,0.974429,0.910690,0.981466,1,ema
2,2,0.989132,0.905706,0.990119,2,ema
3,3,0.989916,0.953036,0.989627,3,ema
4,4,0.967023,0.940195,0.962025,4,ema


## 6. Original-only test inference
One GPU model at a time. No flips, no second forward pass. Per-fold caches validate
checkpoint SHA and ordered IDs and can be reused after rerunning this section.
The original 224 predictions are read from the attached backup.

In [ ]:
assert not missing_folds, f'Finish all five folds first: {missing_folds}'
ensemble_config = json.loads((CV_DIR / 'ensemble_config.json').read_text())
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class TestImages(Dataset):
    def __len__(self):
        return len(sample)
    def __getitem__(self, index):
        image_id = sample.iloc[index].image_id
        with Image.open(DATA_DIR / 'test' / f'{image_id}.jpg') as image:
            return eval_transform(image=np.array(image.convert('RGB')))['image']

_, eval_transform = make_transforms()
test_loader = DataLoader(TestImages(), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=DEVICE.type == 'cuda')
mean_original = np.zeros((len(sample), len(LABEL_COLS)), dtype=np.float64)
test_ids = sample.image_id.to_numpy(dtype=str)
for fold in range(5):
    checkpoint_path = CV_DIR / f'fold_{fold}' / 'best.pt'
    checkpoint_hash = file_digest(checkpoint_path)
    assert checkpoint_hash == ensemble_config['checkpoint_hashes'][str(fold)]
    cache_path = TEST_OUTPUT_DIR / f'fold_{fold}_test_original.npz'
    cache_ok = False
    if cache_path.is_file():
        try:
            with np.load(cache_path, allow_pickle=False) as cache:
                original = cache['original']
                cache_ok = (str(cache['checkpoint_hash'].item()) == checkpoint_hash
                            and np.array_equal(cache['image_ids'], test_ids)
                            and np.array_equal(cache['label_cols'], np.array(LABEL_COLS))
                            and original.shape == mean_original.shape and np.isfinite(original).all()
                            and ((original >= 0) & (original <= 1)).all())
        except (OSError, ValueError, KeyError):
            cache_ok = False
    if not cache_ok:
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        cfg = checkpoint['config']
        assert cfg['cv_fingerprint'] == cv_fingerprint and cfg['fold'] == fold
        assert cfg['model_name'] == MODEL_NAME and cfg['label_cols'] == LABEL_COLS
        assert cfg['preprocessing'] == preprocessing
        model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(LABEL_COLS)).to(
            device=DEVICE, memory_format=MEMORY_FORMAT)
        model.load_state_dict(checkpoint['model'], strict=True)
        model.eval()
        del checkpoint
        parts = []
        with torch.inference_mode():
            for images in tqdm(test_loader, desc=f'fold {fold}: test, no TTA'):
                images = images.to(DEVICE, non_blocking=True, memory_format=MEMORY_FORMAT)
                with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    logits = model(images)
                parts.append(logits.float().sigmoid().cpu().numpy())
        original = np.concatenate(parts)
        assert original.shape == mean_original.shape and np.isfinite(original).all()
        assert ((original >= 0) & (original <= 1)).all()
        temporary = cache_path.with_suffix('.tmp.npz')
        np.savez_compressed(temporary, original=original, image_ids=test_ids,
                            label_cols=np.array(LABEL_COLS), checkpoint_hash=np.array(checkpoint_hash))
        temporary.replace(cache_path)
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    else:
        print(f'Fold {fold}: reusing verified original-only test predictions.')
    mean_original += original.astype(np.float64) / 5
assert np.isfinite(mean_original).all() and ((mean_original >= 0) & (mean_original <= 1)).all()
probability_frame = pd.DataFrame(mean_original, columns=LABEL_COLS)
probability_frame.insert(0, 'image_id', sample.image_id.to_numpy())
probability_frame.to_csv(TEST_OUTPUT_DIR / 'test_probabilities_original.csv', index=False)
save_json(ensemble_config, TEST_OUTPUT_DIR / 'ensemble_config.json')


fold 4: test, no TTA: 100%|██████████| 1168/1168 [02:20<00:00,  8.29it/s]


## 7. Equal 224/320 blend, thresholds and submissions
Reuse the established 50/50 probability mean and OOF threshold search. No test labels
and no leaderboard weight search. Thresholds are fitted to the corresponding original-only
OOF mean. The main candidate is `submission_blend_224_320.csv`.

In [ ]:
from IPython.display import FileLink
BLEND_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
source_original = source_oof[[label + '_prob' for label in LABEL_COLS]].to_numpy(dtype=np.float64)
fine_original = oof[[label + '_prob' for label in LABEL_COLS]].to_numpy(dtype=np.float64)
assert source_oof.image_id.tolist() == oof.image_id.tolist() == frame.image_id.tolist()
assert np.array_equal(source_oof[LABEL_COLS].to_numpy(), oof[LABEL_COLS].to_numpy())
blend_oof = (source_original + fine_original) / 2
blend_thresholds = tune_thresholds(targets, blend_oof)
blend_test = (source_test_p + mean_original) / 2
blend_report, blend_table = metric_report(targets, blend_oof, blend_thresholds)
display(blend_table)
print('Blend fitted OOF metrics (calibration, optimistic):', blend_report)

def export_submission(directory, filename, values, thresholds):
    submission = sample.copy()
    submission[LABEL_COLS] = (values >= np.asarray(thresholds)).astype(np.int32)
    assert len(submission) == len(test) and not submission.isna().any().any()
    assert submission[LABEL_COLS].isin([0, 1]).all().all()
    path = directory / filename
    submission.to_csv(path, index=False)
    print('Saved:', path)
    display(FileLink(str(path)))

export_submission(TEST_OUTPUT_DIR, 'submission_cv320.csv', mean_original, all_thresholds['original'])
export_submission(BLEND_OUTPUT_DIR, 'submission_blend_224_320.csv', blend_test, blend_thresholds)
blended_probabilities = pd.DataFrame(blend_test, columns=LABEL_COLS)
blended_probabilities.insert(0, 'image_id', sample.image_id.to_numpy())
blended_probabilities.to_csv(BLEND_OUTPUT_DIR / 'test_probabilities_blend_original.csv', index=False)
save_json({'weights': {'224': 0.5, '320': 0.5}, 'label_cols': LABEL_COLS,
           'base_cv_fingerprint': source_manifest['cv_fingerprint'], 'fine_cv_fingerprint': cv_fingerprint,
           'thresholds': {'original': blend_thresholds.tolist()}, 'tta': False}, BLEND_OUTPUT_DIR / 'blend_config.json')
pd.DataFrame([{'candidate': '320', **summary.loc[summary.threshold_source == 'fitted_oof'].iloc[0].to_dict()},
              {'candidate': 'blend_224_320', **blend_report}]).to_csv(BLEND_OUTPUT_DIR / 'oof_comparison.csv', index=False)


,positives,predicted positives,f1,precision,recall,roc_auc
label,,,,,,
Cat,5077,5061,0.996054,0.997629,0.994485,0.999765
Dog,5025,5036,0.994136,0.993050,0.995224,0.999773
Moris,56,54,0.963636,0.981481,0.946429,0.999835
Motya,42,44,0.976744,0.954545,1.000000,0.999982
Biatrix,41,39,0.975000,1.000000,0.951220,0.999792


Blend fitted OOF metrics (calibration, optimistic): {'macro_f1': 0.9811141540168272, 'macro_roc_auc': 0.9998294048722862}
Saved: /kaggle/working/convnext_cv_320_fast_ensemble/submission_cv320.csv


/kaggle/working/convnext_cv_320_fast_ensemble/submission_cv320.csv

Saved: /kaggle/working/convnext_blend_224_320_fast/submission_blend_224_320.csv


/kaggle/working/convnext_blend_224_320_fast/submission_blend_224_320.csv

## 8. Download the compact model backup before stopping
Five best checkpoints plus config, split, OOF, predictions and submissions. No `last.pt`
or W&B logs; this is sufficient for inference and for the ResNet probability ensemble.
Keep the old 224 backup separately. Creating a ZIP in working does not make it persistent.
Download it from Kaggle Output and verify the five best.pt files, or save a version
with output files included before ending the session.

In [ ]:
create_inference_backup()
from zipfile import ZipFile
with ZipFile(BACKUP_PATH) as archive:
    names = set(archive.namelist())
    for fold in range(5):
        assert f'{CV_DIR.name}/fold_{fold}/best.pt' in names
    assert f'{CV_DIR.name}/ensemble_config.json' in names
    assert f'{TEST_OUTPUT_DIR.name}/test_probabilities_original.csv' in names
print(f'Notebook elapsed: {(perf_counter() - RUN_STARTED) / 60:.1f} min; download time is additional.')
display(FileLink(str(BACKUP_PATH)))
print('Refresh Kaggle Output, DOWNLOAD the archive, and verify it opens before stopping the session.')


Backup refreshed: /kaggle/working/convnext_320_fast_inference_backup.zip (0.53 GiB). Download it before ending the session.
Notebook elapsed: 81.9 min; download time is additional.


/kaggle/working/convnext_320_fast_inference_backup.zip

Refresh Kaggle Output, DOWNLOAD the archive, and verify it opens before stopping the session.
